<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/mid_module_exam_02_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Problem Statement
Build a small character-level LSTM next-character model in PyTorch. The model must take a sequence of character indices, embed them, pass them through an LSTM, and produce per-timestep logits over the vocabulary. Train it on a tiny inline corpus so that the loss decreases over a handful of epochs.

Dataset
Use the following inline corpus and build the vocabulary from its unique characters (including spaces and punctuation):

corpus = "deep learning is powerful and fun"
From the corpus, construct:

A sorted list of unique characters as the vocabulary.
Two dictionaries: char_to_idx and idx_to_char.
Training pairs by sliding a window of length 5 across the corpus, where the input is the 5 consecutive characters and the target is the next character at each position (i.e., for input length T the target is also length T, shifted by one).
Tasks
Construct the vocabulary, the two index mappings, and the input/target tensors as PyTorch LongTensors.
Define a CharLSTM PyTorch module with:
An embedding layer of dimension 16.
A single-layer LSTM with hidden size 64.
A final linear layer mapping the LSTM hidden states to vocabulary logits.
A forward(x) that returns logits of shape (batch, sequence_length, vocab_size).
Write a training loop using Adam and cross-entropy loss across all positions (you may flatten predictions and targets to 2D and 1D respectively for the loss). Train for at least 20 epochs and print the loss every 5 epochs.
After training, pick the input sequence "deep " (the first 5 characters), run a single forward pass, and print the predicted next character (the argmax over the final timestep's logits).
Expected Output
Loss should decrease over the printed epochs, and the model should produce a plausible next character given the prefix "deep " (any character from the corpus that follows "deep " in some training pair counts as plausible).

Submission Format
Paste your complete PyTorch implementation in the input box (vocab construction, training pairs, model class, training loop, and the final prediction print).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Inline corpus
corpus = "deep learning is powerful and fun"

# ---------------------------------------------------
# Vocabulary construction
# ---------------------------------------------------
vocab = sorted(list(set(corpus)))
vocab_size = len(vocab)

char_to_idx = {ch: idx for idx, ch in enumerate(vocab)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

print("Vocabulary:", vocab)
print("Vocabulary Size:", vocab_size)

# ---------------------------------------------------
# Create training data using sliding window
# Input length = 5
# Target is shifted by one character
# ---------------------------------------------------
seq_length = 5

inputs = []
targets = []

for i in range(len(corpus) - seq_length):
    input_seq = corpus[i:i + seq_length]
    target_seq = corpus[i + 1:i + seq_length + 1]

    inputs.append([char_to_idx[ch] for ch in input_seq])
    targets.append([char_to_idx[ch] for ch in target_seq])

# Convert to tensors
X = torch.tensor(inputs, dtype=torch.long)
Y = torch.tensor(targets, dtype=torch.long)

print("Input tensor shape:", X.shape)
print("Target tensor shape:", Y.shape)

# ---------------------------------------------------
# Define CharLSTM model
# ---------------------------------------------------
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_size=64):
        super(CharLSTM, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x shape: (batch, seq_length)

        embedded = self.embedding(x)
        # embedded: (batch, seq_length, embed_dim)

        lstm_out, _ = self.lstm(embedded)
        # lstm_out: (batch, seq_length, hidden_size)

        logits = self.fc(lstm_out)
        # logits: (batch, seq_length, vocab_size)

        return logits

# ---------------------------------------------------
# Initialize model, loss, optimizer
# ---------------------------------------------------
model = CharLSTM(vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ---------------------------------------------------
# Training loop
# ---------------------------------------------------
epochs = 20

for epoch in range(1, epochs + 1):

    model.train()

    optimizer.zero_grad()

    logits = model(X)

    # Flatten for cross-entropy
    # logits: (batch * seq_length, vocab_size)
    # targets: (batch * seq_length)
    loss = criterion(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# ---------------------------------------------------
# Final prediction using prefix "deep "
# ---------------------------------------------------
model.eval()

test_seq = "deep "
test_input = torch.tensor(
    [[char_to_idx[ch] for ch in test_seq]],
    dtype=torch.long
)

with torch.no_grad():
    output_logits = model(test_input)

    # Take logits from final timestep
    final_timestep_logits = output_logits[0, -1]

    predicted_idx = torch.argmax(final_timestep_logits).item()

    predicted_char = idx_to_char[predicted_idx]

print(f'\nInput Sequence: "{test_seq}"')
print("Predicted next character:", predicted_char)

Vocabulary: [' ', 'a', 'd', 'e', 'f', 'g', 'i', 'l', 'n', 'o', 'p', 'r', 's', 'u', 'w']
Vocabulary Size: 15
Input tensor shape: torch.Size([28, 5])
Target tensor shape: torch.Size([28, 5])
Epoch 5, Loss: 2.3360
Epoch 10, Loss: 1.6251
Epoch 15, Loss: 0.8970
Epoch 20, Loss: 0.4629

Input Sequence: "deep "
Predicted next character: l


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

corpus = "deep learning is powerful and fun"

# --- Vocabulary and mappings ---
chars = sorted(set(corpus))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
V = len(chars)

# --- Training pairs: window of length 5, target shifted by one ---
T = 5
encoded = [char_to_idx[c] for c in corpus]
inputs, targets = [], []
for i in range(len(encoded) - T):
    inputs.append(encoded[i:i + T])
    targets.append(encoded[i + 1:i + 1 + T])
X = torch.tensor(inputs, dtype=torch.long)   # (N, T)
Y = torch.tensor(targets, dtype=torch.long)  # (N, T)

class CharLSTM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 16, hidden: int = 64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.lstm  = nn.LSTM(emb_dim, hidden, batch_first=True)
        self.head  = nn.Linear(hidden, vocab_size)

    def forward(self, x):                  # x: (B, T)
        e = self.embed(x)                  # (B, T, E)
        h, _ = self.lstm(e)                # (B, T, H)
        logits = self.head(h)              # (B, T, V)
        return logits

model = CharLSTM(V)
opt = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

epochs = 25
for epoch in range(1, epochs + 1):
    opt.zero_grad()
    logits = model(X)                          # (B, T, V)
    loss = loss_fn(logits.reshape(-1, V), Y.reshape(-1))
    loss.backward()
    opt.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch}: loss = {loss.item():.4f}")

# --- Predict next character for prefix "deep " ---
prefix = "deep "
prefix_ids = torch.tensor([[char_to_idx[c] for c in prefix]], dtype=torch.long)  # (1, 5)
with torch.no_grad():
    logits = model(prefix_ids)              # (1, 5, V)
    next_idx = int(logits[0, -1].argmax())  # last timestep argmax
print("Predicted next character:", repr(idx_to_char[next_idx]))


Epoch 5: loss = 2.3645
Epoch 10: loss = 1.6722
Epoch 15: loss = 1.0065
Epoch 20: loss = 0.5454
Epoch 25: loss = 0.3329
Predicted next character: 'l'


roblem Statement
You are building a small denoising autoencoder for grayscale 28x28 MNIST-style images using PyTorch. The encoder must compress a noisy flattened image into a latent vector, and the decoder must reconstruct a clean image. Train the model with mean squared error so that the latent representation captures the informative content while filtering out noise.

Dataset
Assume a PyTorch tensor x_clean of shape (N, 28, 28) containing clean grayscale images with pixel values already scaled to the range [0, 1]. Generate the noisy training input inline by adding Gaussian noise:

import torch
noise = 0.3 * torch.randn_like(x_clean)
x_noisy = torch.clamp(x_clean + noise, 0.0, 1.0)
Use x_noisy as the input to the encoder and x_clean as the reconstruction target. You may keep N small (e.g. 1024 samples) for the purposes of this question.

Tasks
Define a DenoisingAutoencoder PyTorch module with:
An encoder built from fully connected (linear) layers with ReLU activations that flattens the 28x28 input and compresses it to a latent vector of size 64.
A decoder built from fully connected layers with ReLU activations that maps the latent vector back to a 28x28 reconstruction (use a final sigmoid so outputs are in [0, 1]).
A forward(x) method that flattens the input, encodes, decodes, and reshapes the output back to (batch, 28, 28).
Write a training loop that:
Uses the Adam optimizer.
Uses mean squared error loss between the reconstruction and x_clean.
Runs for at least 5 epochs over mini-batches drawn from x_noisy / x_clean.
Prints the average loss per epoch.
After training, take any single sample, pass its noisy version through the model, and print the shape of the encoder's latent vector to confirm it is (1, 64).
Expected Output
The code should run end-to-end and produce decreasing average training loss across epochs, plus a printed latent-vector shape of (1, 64) for the inspected sample.

Submission Format
Paste your complete PyTorch implementation (model class + training loop + latent-shape print) as a single code snippet in the input box.



In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------------------------------------------------
# Create dummy MNIST-style clean dataset
# Shape: (N, 28, 28)
# Pixel values in [0, 1]
# ---------------------------------------------------
N = 1024

x_clean = torch.rand(N, 28, 28)

# ---------------------------------------------------
# Add Gaussian noise
# ---------------------------------------------------
noise = 0.3 * torch.randn_like(x_clean)
x_noisy = torch.clamp(x_clean + noise, 0.0, 1.0)

# ---------------------------------------------------
# Create DataLoader
# ---------------------------------------------------
dataset = TensorDataset(x_noisy, x_clean)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# ---------------------------------------------------
# Define Denoising Autoencoder
# ---------------------------------------------------
class DenoisingAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super(DenoisingAutoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, latent_dim)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Flatten input
        x = x.view(x.size(0), -1)

        # Encode
        latent = self.encoder(x)

        # Decode
        reconstruction = self.decoder(latent)

        # Reshape back to image format
        reconstruction = reconstruction.view(-1, 28, 28)

        return reconstruction

# ---------------------------------------------------
# Initialize model, optimizer, loss
# ---------------------------------------------------
model = DenoisingAutoencoder(latent_dim=64)

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------------------------------
# Training loop
# ---------------------------------------------------
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for noisy_imgs, clean_imgs in dataloader:

        optimizer.zero_grad()

        # Forward pass
        outputs = model(noisy_imgs)

        # Compute reconstruction loss
        loss = criterion(outputs, clean_imgs)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(dataloader)

    print(f"Epoch [{epoch+1}/{epochs}], Average Loss: {avg_loss:.6f}")

# ---------------------------------------------------
# Inspect latent vector shape for one sample
# ---------------------------------------------------
model.eval()

sample_noisy = x_noisy[0].unsqueeze(0)  # Shape: (1, 28, 28)

with torch.no_grad():

    # Flatten sample
    sample_flat = sample_noisy.view(1, -1)

    # Get latent vector
    latent_vector = model.encoder(sample_flat)

    # Reconstruction
    reconstructed = model(sample_noisy)

print("\nLatent vector shape:", latent_vector.shape)

Epoch [1/5], Average Loss: 0.083386
Epoch [2/5], Average Loss: 0.083285
Epoch [3/5], Average Loss: 0.083254
Epoch [4/5], Average Loss: 0.083229
Epoch [5/5], Average Loss: 0.083192

Latent vector shape: torch.Size([1, 64])


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# --- Toy clean data (replace with real MNIST tensor in [0,1] in production) ---
N = 1024
x_clean = torch.rand(N, 28, 28)  # already in [0, 1]

# --- Construct noisy input ---
noise = 0.3 * torch.randn_like(x_clean)
x_noisy = torch.clamp(x_clean + noise, 0.0, 1.0)

class DenoisingAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 128),     nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256),        nn.ReLU(),
            nn.Linear(256, 28 * 28),
            nn.Sigmoid(),  # outputs in [0, 1]
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out.view(-1, 28, 28)

model = DenoisingAutoencoder()
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

loader = DataLoader(TensorDataset(x_noisy, x_clean), batch_size=64, shuffle=True)

epochs = 5
for epoch in range(1, epochs + 1):
    total, n_batches = 0.0, 0
    for xn, xc in loader:
        opt.zero_grad()
        pred = model(xn)
        loss = loss_fn(pred, xc)
        loss.backward()
        opt.step()
        total += loss.item(); n_batches += 1
    print(f"Epoch {epoch}: avg loss = {total / n_batches:.4f}")

# Latent shape check
sample = x_noisy[:1]
with torch.no_grad():
    latent = model.encoder(sample)
print("Latent shape:", tuple(latent.shape))  # (1, 64)


Epoch 1: avg loss = 0.0834
Epoch 2: avg loss = 0.0833
Epoch 3: avg loss = 0.0833
Epoch 4: avg loss = 0.0833
Epoch 5: avg loss = 0.0832
Latent shape: (1, 64)
